# Predicción de riesgo cardiovascular a 10 años — Modelos de machine learning, evaluación y discución

**Entrega 2 (preprocesamiento, metodología, modelos, análisis y discusión)**

Este notebook continua el trabajo de la Parte 1 (`01-eda.ipynb`) que analisa y explora los datos del dataset de *Framinghan*. Para este notebook se pondrá en practica las decisiones tomadas anteriormente y se crearán modelos de machine learning que resuelvan el problema, posteriormente se evaluará el resultado y se discutirán las conclusiones. El notebook se organiza en [] secciones: 

## 0. El análisis de la Parte 1

Del EDA realizado en la Parte 1 se obtuvo lo siguiente:

* **Tamaño del dataset y objetivo.** 4 240 participantes, 15 variables predictores  y una variable 
  objetivo (`TenYearCHD`) que indica la presencia o ausencia de una enfermedad coronaria en 10 años.
* **Desbalance** De los 4 240 participantes el 15,19 % desarrollo enfermedad coronaria. Es decir, en el dataset
    hay casi 6 personas por 1 persona enferma. Predecir siempre "sin enfermedad" da 84,8% de exactitud, por lo 
    que la exactitud **no** sirve como critetio de éxito.
* **Faltantes.** 13,73 % de las filas contiene al menos un faltante, la variable con más faltantes es `glucose`
  (9,15 %). 
* **Atípicos.** Los valores atípicos son clínicamente posibles (glucosa hasta 394 mg/dL, IMC hasta 56,8, sistólica
  hasta 295 mm Hg), por lo que se van a **conservar**.
* **Redundancia entre predictores.** `sysBP`–`diaBP` (0,78), `cigsPerDay`–`currentSmoker` (0,77),
  `sysBP`–`prevalentHyp` (0,70), `glucose`–`diabetes` (0,62). Son variables que representan resumenes clinimos parcialmente
  redundantes.
* **Correlación individual baja.** Ninguna variable supera una correlación de 0,23 con la variable objetivo


## 1. Carga de Datos 

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nb_utils import use_style, COLORBLIND_CYCLE   # estilo de gráficos del curso
from sklearn.model_selection import train_test_split, StratifiedKFold

use_style("notes")
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 20)

DATA = Path("../data/framingham.csv")
FIG = Path("../figures")
FIG.mkdir(exist_ok=True)

TARGET = "TenYearCHD"
NUM = ["age", "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose", "cigsPerDay"]
BIN = ["male", "currentSmoker", "BPMeds", "prevalentStroke", "prevalentHyp", "diabetes"]
ORD = ["education"]
FEATURES = NUM + BIN + ORD
SEED = 3        # unica semilla
TEST_SIZE = 0.2 # particion 80/20
N_FOLDS = 5     # crossvalidation 5 plieges

ETIQUETAS = {
    "age": "edad (años)", "totChol": "colesterol total (mg/dL)", "sysBP": "presión sistólica (mm Hg)",
    "diaBP": "presión diastólica (mm Hg)", "BMI": "índice de masa corporal (kg/m²)",
    "heartRate": "frecuencia cardíaca en reposo (lpm)", "glucose": "glucosa (mg/dL)",
    "cigsPerDay": "cigarrillos por día", "male": "sexo (1 = hombre)", "currentSmoker": "fumador activo",
    "BPMeds": "medicación antihipertensiva", "prevalentStroke": "ACV previo", "prevalentHyp": "hipertensión previa",
    "diabetes": "diabetes", "education": "nivel educativo (1-4)", "TenYearCHD": "enfermedad coronaria a 10 años",
}
C0, C1 = "#0072B2", "#D55E00"       # color clase 0 (sin evento) / clase 1 (con evento)

def guardar(fig, nombre):
    fig.savefig(FIG / nombre)
    plt.show()

raw = pd.read_csv(DATA)
print("dataset:", raw.shape[0], "participantes x", raw.shape[1], "columnas")

dataset: 4240 participantes x 16 columnas


## 2. Partición de los datos

### 2.1 Partición train / test

La partición de los datos se hace **antes de cualquier modificación**. Se realiza para evitar
el *data leakage*: cualquier estadistica que se calcule después debe salir solo del conjunto de 
entrenamiento. Si no se realizará la particion, la información de los datos de prueba se filtraría
hacia el entrenamiento y las métricas saldrían optimistas.

A considerar:
* **Estratificación**: para que ambos subconjuntos conserven la proporción de aproximadamente 15 % de positivos 
  se necesita estratificar por la variable objetivo. Sin estratificar, con tan pocos positivos, el conjunto de prueba podría
  quedar con una proporción muy distinta.
* **80 % train / 20 % test**, como se propuso en la Parte 1.
* El conjunto de test **no se vuelve a tocar**, ni para elegir hiperparámetros, 
  ni para comparar modelos, etc. Estas deciciones se realizan con cross validation sobre el conjunto de train

In [6]:
X = raw[FEATURES].copy()
y = raw[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=TEST_SIZE,stratify=y,random_state=SEED,)
print(f"train: {X_train.shape[0]} filas  |  test: {X_test.shape[0]} filas")

train: 3392 filas  |  test: 848 filas


### 2.2 CrossValidation Estratificado

Como se menciono para comparar modelos y ajustar hiperparámetros se usa **crossvalidation de 5
pliegues** sobre el conjunto de train, por dos razones:

* **Estratificada**, porque con 15 % de positivos un pliegue aleatorio podría quedar con muy pocos
  casos de la clase minoritaria y producir métricas inestables.
* **Con shuffle y semilla fija** (`shuffle=True`, `random_state=SEED`), para que todos los
  modelos se evalúen sobre **exactamente los mismos pliegues**. Así las diferencias entre modelos
  reflejan al modelo y no el azar de la partición.


In [7]:
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# tamaño y variable objetiva en cada pliegue, solo para verificar que esten balanceados
filas = []
for k, (idx_tr, idx_val) in enumerate(cv.split(X_train, y_train), start=1):
    filas.append({"pliegue": k,
                  "n entrena": len(idx_tr), "n valida": len(idx_val),
                  "% positivos (entrena)": 100 * y_train.iloc[idx_tr].mean(),
                  "% positivos (valida)":  100 * y_train.iloc[idx_val].mean()})
pd.DataFrame(filas).set_index("pliegue").round(2)

,n entrena,n valida,% positivos (entrena),% positivos (valida)
pliegue,,,,
1,2713,679,15.19,15.17
2,2713,679,15.19,15.17
3,2714,678,15.18,15.19
4,2714,678,15.18,15.19
5,2714,678,15.18,15.19


## 3. Preprocesamiento e ingenería de características

Esta sección convierte `X_train` en una matriz que pueda ser fácilmente usada por los modelos
de machine learning. Cada decisión para procesar lo datos viene de haber realizado el EDA en la Parte 1.
Todo el procesamiento está estructurado mediante un `Pipeline` de *scikit-learn*, lo que garantiza la ausencia de data leakage y permite 
reutilizar la misma cadena en pasos posteriores


### 3.1 Resumen

| Variables | ¿Qué se hizo? | Justificación |
| --- | --- | --- |
| `glucose`, `totChol`, `BMI` ( datos faltantes) | Se rellenan con la mediana de su mismo sexo  | Los datos que faltan se parecen más a los de su grupo por sexo que a los de toda la muestra. |
| `cigsPerDay` ( datos faltantes) | Se pone 0 si la persona no fuma. Si fumará, se usa la mediana de los fumadores | Si la persona no fuma, su `cigsPerDay` debe ser 0 |
| `education`, `BPMeds` (datos faltantes) | Se rellenan con el valor más frecuente | Son variables categóricas así que asignarle el valor más frecuente no cambia mucho el análisis |
| `heartRate` (datos faltantes) | Se rellena con la mediana de todos | Esta variable solo falta un dato, asi que no influye mucho |
| Variables numéricas | Se escalan con `RobustScaler` | Deja todas en una escala parecida usando la mediana y el rango intercuartílico. Así los valores muy altos de `glucose`  no distorsionan al resto |
| Variables binarias y `education` | No se transforman | Las binarias ya están en 0/1, y `education` se deja como número de 1 a 4 para respetar su orden |
| Presión de pulso (variable nueva, `pulsePressure`) | Se crea como `sysBP` − `diaBP` | Mide la rigidez de las arterias principales. Resumirá en un solo número la diferencia entre las dos presiones |

### 3.2 Presión de pulso

La **presión de pulso** es un indicador clínico de la rigidez de las arterias. Se calcula fila por fila, 
y es una simple resta de 2 variables ya existentes en el dataset: `sysBP` y `diaBP`


In [8]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

## Integracion con el pipeline
class anadirPresionPulso(BaseEstimator, TransformerMixin):
    # Agrega pulsePressure = sysBP - diaBP.
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["pulsePressure"] = X["sysBP"] - X["diaBP"]
        return X


# temp es una copia de X_train agregando la nueva columna
tmp = X_train.assign(pulsePressure=X_train["sysBP"] - X_train["diaBP"])
columnas = ["sysBP","diaBP","pulsePressure"]

# se crea un dataset para analizar la tabla de correlaciones
pd.DataFrame({
    # calculamos la correlacion de sysBP, diaBP y pulsePressure con la variable objetivo
    "corr con TenYearCHD":tmp[columnas].corrwith(y_train),
    
    # calculamos la correlación de las 3 variables contra sysBP.
    "corr con sysBP":tmp[columnas].corrwith(tmp["sysBP"]),
    
    # calculamos la correlación de las 3 variables contra diaBP.
    "corr con diaBP":tmp[columnas].corrwith(tmp["diaBP"]),
}).round(3)

,corr con TenYearCHD,corr con sysBP,corr con diaBP
sysBP,0.222,1.000,0.779
diaBP,0.151,0.779,1.000
pulsePressure,0.209,0.864,0.357


La presión de pulso tiene una correlación de 0,209 con `TenYearCHD`, casi igual a la de `sysBP` (0,222) y mayor que la de `diaBP` (0,151). Se parece poco a `diaBP` (0,357), pero bastante a `sysBP` (0,864), por lo que no es una variable independiente: resume la diferencia entre las dos presiones. 

### 3.3 Valores faltantes

En lugar de eliminar filas por valores faltantes, se utilizará un transformador propio para imputarlos. Este aprenderá las medianas y modas únicamente del conjunto recibido en fit, lo que garantiza que dentro de la validación cruzada solo se calculen a partir de los pliegues de entrenamiento.

In [9]:
class solucionValoresFaltantes(BaseEstimator, TransformerMixin):
    # glucose, totChol, BMI: mediana por sexo
    # cigsPerDay: 0 si no fuma; mediana de los fumadores si fuma
    # education, BPMeds : moda
    # heartRate: mediana global

    def __init__(self, cols_por_sexo=("glucose", "totChol", "BMI"),
                 cols_moda=("education", "BPMeds"), cols_global=("heartRate",)):
        self.cols_por_sexo = cols_por_sexo
        self.cols_moda = cols_moda
        self.cols_global = cols_global

    def fit(self, X, y=None):
        # Calcula las medianas agrupadas por la variable 'male' para cada columna de interes
        self.med_sexo_ = {c: X.groupby("male")[c].median().to_dict() for c in self.cols_por_sexo}
        # Guarda medianas globales por si falla la busqueda por sexo
        self.med_global_ = {c: X[c].median() for c in (*self.cols_por_sexo, *self.cols_global)}
        #Calcula la moda para variables categóricas
        self.moda_ = {c: X[c].mode().iloc[0] for c in self.cols_moda}
        #Calcula la mediana de cigarrillos consumidos (solo para los que fuman)
        self.cigs_fumador_ = X.loc[X["currentSmoker"] == 1, "cigsPerDay"].median()
        return self

    def transform(self, X):
        X = X.copy()
        # Imputa por sexo, si no por la mediana global
        for c in self.cols_por_sexo:
            X[c] = X[c].fillna(X["male"].map(self.med_sexo_[c])).fillna(self.med_global_[c])
        # Imputacion condicional para os cigarrillos
        falta = X["cigsPerDay"].isna()
        X.loc[falta & (X["currentSmoker"] == 0), "cigsPerDay"] = 0
        X.loc[falta & (X["currentSmoker"] == 1), "cigsPerDay"] = self.cigs_fumador_
        # mediana global
        for c in self.cols_global:
            X[c] = X[c].fillna(self.med_global_[c])
        # moda
        for c in self.cols_moda:
            X[c] = X[c].fillna(self.moda_[c])
        return X

In [10]:
imp = solucionValoresFaltantes().fit(X_train)

aprendidos = pd.DataFrame(imp.med_sexo_).rename(index={0: "mujer", 1: "hombre"}).round(1)
aprendidos.loc["global"] = {c: round(imp.med_global_[c], 1) for c in aprendidos.columns}
print("medianas aprendidas (train):")
print(aprendidos.to_string())
print("cigsPerDay (mediana de fumadores):", imp.cigs_fumador_)
print("modas:", imp.moda_)

medianas aprendidas (train):
        glucose  totChol   BMI
mujer      78.0    236.0  24.8
hombre     78.0    230.0  26.0
global     78.0    233.0  25.4
cigsPerDay (mediana de fumadores): 20.0
modas: {'education': np.float64(1.0), 'BPMeds': np.float64(0.0)}


### 3.4 Codificación y escalamiento

* **Binarias** (las 6 variables): se dejarán tal cual
* **education**: variable ordinal de 1 a 4, se mantiene como una sola columna numérica. Esto preserva el orden implícito de los niveles y evita añadir tres columnas dummy sobre aproximadamente 3 400 observaciones.
* **Numéricas continuas** (las 8 originales + plusePressure): se utiliza RobustScaler, el cual centra con la mediana y escala mediante el rango intercuartílico. Se prefiere sobre StandardScaler debido a que se conservan los valores atípicos.

In [11]:
escalado = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), NUM + ["pulsePressure"]),
        ("pasan", "passthrough", BIN+ORD),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

# preprocesamiento completo: anadirPresionPulso -> solucionValoresFaltantes -> escalado(este paso)
preproc = Pipeline([
    ("pulso", anadirPresionPulso()),
    ("imputar", solucionValoresFaltantes()),
    ("escalar", escalado),
])
preproc

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('pulso', ...), ('imputar', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,cols_por_sexo,"('glucose', ...)"
,cols_moda,"('education', ...)"
,cols_global,"('heartRate',)"
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('pasan', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"s

### 3.5 Verificación del preprocesamiento

Se ajusta el pipeline **solo con `X_train`** y se aplica a train y a test, para comprobar que corre de principio a fin y que no quedan valores faltantes. Más adelante, `preproc` se encadena con cada modelo y la validación cruzada lo vuelve a ajustar en cada pliegue.

In [12]:
Xtr = preproc.fit(X_train).transform(X_train)
Xte = preproc.transform(X_test)

assert Xtr.isna().sum().sum() == 0 and Xte.isna().sum().sum() == 0
print("train:", Xtr.shape, "  test:", Xte.shape)
print("columnas:", list(Xtr.columns))

train: (3392, 16)   test: (848, 16)
columnas: ['age', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'cigsPerDay', 'pulsePressure', 'male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'education']


## 4. Metodología y modelos

Esta sección presenta los modelos que se compararán y justifica su elección. Todos reciben los datos a través del mismo `preproc` y se evalúan con el mismo esquema de validación cruzada `cv` , por lo que las diferencias entre ellos se deben al modelo .

### 4.1 Baselines

Un baseline es el punto de referencia mínimo: si un modelo no lo supera con claridad, no aporta valor. Se usan dos:

| Baseline | Qué hace | Para qué sirve |
| --- | --- | --- |
| **Trivial** (`DummyClassifier`) | Predice siempre la proporción de positivos del entrenamiento (15 %), sin mirar ninguna variable | La exactitud ya sería ~85 % por eso es que no se usa como criterio |
| **Regresión logística solo con `age`** | Un modelo lineal con una única variable, la de mayor correlación con el objetivo | Mide cuánto se gana al usar las demás variables frente a la mejor señal individual |

In [14]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# --- baselines ---
baseline_trivial = Pipeline([
    ("preproc", preproc),
    ("modelo", DummyClassifier(strategy="prior")),
])

baseline_age = Pipeline([
    ("solo_age", ColumnTransformer([("age", RobustScaler(), ["age"])])),
    ("modelo", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

### 4.2 Modelos comparados

Se comparan dos enfoques de naturaleza distinta, para ver si la relación entre las variables y la variable objetivo es esencialmente lineal o si hay interacciones que un modelo más flexible aprovecha.

**Regresión logística regularizada (enfoque lineal).**
- Es el estándar en riesgo clínico, tiene pocos hiperparámetros y sus coeficientes indican qué variables suben o bajan el riesgo.
- *Regularización L2:* se aplica porque el EDA mostró variables parcialmente redundantes. Sin regularización, el modelo repartiría el mismo efecto entre variables correlacionadas.
- *Hiperparámetro principal:* `C`, la inversa de la fuerza de regularización (valores pequeños penalizan más).

**Random Forest (enfoque no lineal).**
- Captura relaciones no lineales e interacciones sin que haya que definirlas a mano, no necesita escalado y es poco sensible a los atípicos que se conservaron.
- *Riesgo y control:* con ~3 400 observaciones, un bosque muy profundo memorizaría ruido. Por eso se restringe su complejidad con árboles poco profundos y hojas grandes.
- *Hiperparámetros principales:* `max_depth`, `min_samples_leaf` y `max_features`.

In [15]:
# --- modelos principales ---
modelo_lr = Pipeline([
    ("preproc", preproc),
    ("modelo", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
])

modelo_rf = Pipeline([
    ("preproc", preproc),
    ("modelo", RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",n_jobs=-1, random_state=SEED)),
])

### 4.3 Manejo del desbalance de clases

Con ~15 % de positivos, un modelo sin ajuste tiende a ignorar la clase minoritaria. Se compararán dos estrategias:

* **`class_weight="balanced"`** (estrategia principal): penaliza más los errores sobre los casos positivos.
* **SMOTE**: genera casos positivos sintéticos. Solo se aplica dentro de cada pliegue de train, nunca sobre los datos de validación ni de prueba, para no contaminar la evaluación.